# IAM hands-on with the AWS CLI

**Goal:** by the end of this lesson you can use the AWS command line (the *CLI*) to create users, groups, roles and policies — and understand *why* each of them exists.

| Part | Topic | Course material |
|---|---|---|
| 0 | Set up the CLI inside the notebook | — |
| 1 | How a CLI command is built (and the most common errors) | — |
| 2 | IAM in 1 minute | UD 01 – 1.1 |
| 3 | Users, groups and roles | UD 02 – 2.1 |
| 4 | Access key and secret access key | UD 02 – 2.2 |
| 5 | MFA (multi-factor authentication) | UD 02 – 2.3 |
| 6 | Policies: what they are, types, creating and testing them | UD 03 – 3.1 to 3.3 |
| 7 | Best practices | UD 03 – 3.4 |
| 8 | Clean-up (delete everything we created) | — |
| 9 | Exercises | — |

> **Before you start:** you need an AWS account and a user with administrator permissions (ideally **not** the *root* user). IAM is free, and at the end we delete everything we created.

### How to use this notebook
- A notebook has **text cells** (like this one) and **code cells**.
- To run a code cell: click it and press **Shift + Enter**.
- Run the cells **in order**, from top to bottom.
- When a line starts with **`!`**, it is a terminal command — exactly what you would type in your computer's terminal, just without the `!`.

---
## Part 0 — Set up the CLI

The **AWS CLI** is a program that lets you give orders to AWS by typing commands instead of clicking around the website (the *console*).

The cell below checks whether the CLI is already installed. If it is not, it installs it (this automatic install only works on Linux, e.g. Google Colab — on your own computer, follow the README).

In [ ]:
!aws --version 2>/dev/null || (curl -s "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o /tmp/awscliv2.zip && unzip -q -o /tmp/awscliv2.zip -d /tmp && /tmp/aws/install --update && aws --version)

### Log in (without leaving your keys in the notebook)

The CLI needs two pieces of information to know **who you are**: the *access key ID* and the *secret access key* (we will see what they are in Part 4).

⚠️ **Never paste your keys straight into a cell.** The notebook is saved and may be shared — anyone with the keys can get into your account. That is why we use `getpass`, which asks for the key in a hidden box and saves nothing in the file.

In [ ]:
import os, json, subprocess, time, random, string
from getpass import getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass("Access key ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("Secret access key: ")
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"   # Ireland. IAM is global, but the CLI asks for a region.
os.environ["AWS_PAGER"] = ""                      # makes the CLI print everything at once, without paging

print("Done! Keys kept only in this session's memory.")

First test: ask AWS **"who am I?"**. If you see the account number and your user, everything is working.

In [ ]:
!aws sts get-caller-identity

Two helper functions we will use later (you do not need to understand their code right now):

In [ ]:
def aws_json(command):
    """Runs a CLI command and returns the answer as a Python dictionary."""
    r = subprocess.run("aws " + command + " --output json", shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
    return json.loads(r.stdout) if r.stdout.strip() else {}

def as_test_user(command):
    """Runs a command using the test user's keys (not yours)."""
    env = dict(os.environ, AWS_ACCESS_KEY_ID=KEY_ID, AWS_SECRET_ACCESS_KEY=SECRET)
    r = subprocess.run(command, shell=True, env=env, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ ALLOWED\n" + r.stdout)
    else:
        print("⛔ DENIED\n" + r.stderr)

# Unique names so we never clash with anything already in the account
SUFFIX = "".join(random.choices(string.ascii_lowercase + string.digits, k=4))
USER   = f"test-student-{SUFFIX}"
GROUP  = f"readers-{SUFFIX}"
ROLE   = f"ec2-role-{SUFFIX}"
POLICY = f"iam-read-{SUFFIX}"
KEY_ID = SECRET = POLICY_ARN = ""
print(USER, GROUP, ROLE, POLICY, sep="\n")

---
## Part 1 — How a CLI command is built

Almost every AWS CLI command follows the same recipe:

```
aws   iam        list-users   --output table
 │     │             │             │
 │     │             │             └── options (how I want the answer, what to apply it to…)
 │     │             └── action (what I want to do)
 │     └── service (which part of AWS I am talking to: iam, s3, ec2…)
 └── the program
```

Read it like a sentence: **"AWS, in the IAM service, list the users, and show them in a table."**

In [ ]:
!aws iam list-users --output table

### The same command, three answer formats
- `--output json` → the "raw" format, good for programs to read.
- `--output table` → good for humans to read.
- `--output text` → plain text, good for scripts.

In [ ]:
!aws iam list-users --output json

### `--query`: keep only the part you care about
AWS answers usually come with a lot of information. `--query` works like a filter: "from the answer, give me only the names".

In [ ]:
!aws iam list-users --query "Users[].UserName" --output text

### Ask the CLI itself for help
No need to memorise commands. Add `help` at the end and the CLI explains:
- `aws iam help` → lists every IAM action
- `aws iam create-user help` → explains one specific action

In [ ]:
!which groff >/dev/null || apt-get -qq install -y groff-base >/dev/null 2>&1   # "help" needs this program on Colab
!aws iam help 2>&1 | grep -o -E "(create|list)-(user|group|role|polic)[a-z-]*" | sort -u

### Where does the CLI look for your keys?
In this order:
1. **Environment variables** (what we did in this notebook).
2. The file **`~/.aws/credentials`**, which is created when you run **`aws configure`** on your computer.

On your own computer the usual way is to run `aws configure` once. If you have more than one account you can create **profiles**: `aws configure --profile work` and then `aws iam list-users --profile work`.

In [ ]:
!aws configure list

### Most common errors (and what they mean)

| Message | In plain words | What to do |
|---|---|---|
| `Unable to locate credentials` | The CLI does not know who you are | Run `aws configure` (or the keys cell) |
| `InvalidClientTokenId` / `SignatureDoesNotMatch` | The key is wrong or was disabled | Check the key, create a new one if needed |
| `AccessDenied` / `UnauthorizedOperation` | AWS knows who you are, but you **are not allowed** | A policy is missing (Part 6) |
| `NoSuchEntity` | The thing you asked for does not exist | Check the name |
| `EntityAlreadyExists` | Something with that name already exists | Use another name |
| `Invalid choice` | Command typed wrong | Use `aws <service> help` |

---
## Part 2 — IAM in 1 minute (UD 01)

**IAM = Identity and Access Management** — the "doorman" of your AWS account.

Think of an office building:
- **Authentication** = showing your badge at the front desk → *"who are you?"*
- **Authorisation** = the badge only opens some doors → *"what are you allowed to do?"*

IAM handles both. It is **global** (it works in every region) and it **costs nothing**.

A summary of your account in numbers:

In [ ]:
!aws iam get-account-summary --query "SummaryMap.{Users:Users,Groups:Groups,Roles:Roles,Policies:Policies,Root_MFA_enabled:AccountMFAEnabled}" --output table

---
## Part 3 — Users, groups and roles (UD 02 – 2.1)

| Concept | Office-building comparison | When to use it |
|---|---|---|
| **User** | A person with their own permanent badge | A person or a program that needs fixed access |
| **Group** | A department: everyone in "finance" opens the same doors | Giving the same permissions to many people at once |
| **Role** | A visitor badge someone borrows for a while | AWS services (e.g. an EC2 server) or temporary access |

The big difference with a **role**: it **has no password and no fixed key**. Whoever "wears" the role gets temporary credentials that expire on their own.

### Create a group and a user, and put the user in the group

In [ ]:
!aws iam create-group --group-name {GROUP}

In [ ]:
!aws iam create-user --user-name {USER}

In [ ]:
!aws iam add-user-to-group --group-name {GROUP} --user-name {USER}
!aws iam get-group --group-name {GROUP} --query "Users[].UserName" --output table

### Create a role
A role needs a **trust policy**: a document that says **who can wear** this role. Here we say: "the EC2 service (AWS servers) can use this role".

The `%%writefile` line saves the content of the cell to a file.

In [ ]:
%%writefile trust-ec2.json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": { "Service": "ec2.amazonaws.com" },
      "Action": "sts:AssumeRole"
    }
  ]
}

In [ ]:
!aws iam create-role --role-name {ROLE} --assume-role-policy-document file://trust-ec2.json --query "Role.Arn" --output text

Now we give the role one permission: read S3 (where files are stored). A server that "wears" this role will be able to read files **without any key stored on it**.

In [ ]:
!aws iam attach-role-policy --role-name {ROLE} --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess
!aws iam list-attached-role-policies --role-name {ROLE} --output table

---
## Part 4 — Access key and secret access key (UD 02 – 2.2)

To use the CLI or a program, a user needs a pair of keys:
- **Access key ID** → like a *username* (starts with `AKIA…`). It can be shown.
- **Secret access key** → this is the *password*. **It is shown only once**, when it is created. If you lose it, you have to create another one.

Each user can have **at most 2 keys** (so one can be swapped for the other without stopping anything).

Let's create a key for the test user. Notice that we hide the secret part on purpose:

In [ ]:
key = aws_json(f"iam create-access-key --user-name {USER}")["AccessKey"]
KEY_ID, SECRET = key["AccessKeyId"], key["SecretAccessKey"]

print("Access key ID:    ", KEY_ID)
print("Secret access key:", SECRET[:4] + "*" * 36, "(hidden on purpose)")

In [ ]:
!aws iam list-access-keys --user-name {USER} --output table

Let's wait a few seconds (AWS takes a moment to spread the new key around) and ask "who am I?" **using the test user's key**:

In [ ]:
time.sleep(10)
as_test_user("aws sts get-caller-identity")

It works: we are now the `test-student`. Keep this in mind: **being able to log in does not mean being allowed to do things.** That comes in Part 6.

---
## Part 5 — MFA: multi-factor authentication (UD 02 – 2.3)

MFA = besides the password, asking for **a second code** that changes every 30 seconds (on your phone, in an app such as Google Authenticator, or on a physical key).

Even if someone steals your password, without your phone they cannot get in.

Turning MFA on is easier in the **console** (because of the QR code). With the CLI we can **check** who has MFA:

In [ ]:
!aws iam list-mfa-devices --user-name {USER} --output table

Empty — the test user has no MFA.

### Audit: who in the account has no MFA?
AWS can generate a **credential report** with every user. It is a great tool for finding security problems.

In [ ]:
import base64, io
import pandas as pd

for _ in range(10):
    if aws_json("iam generate-credential-report").get("State") == "COMPLETE":
        break
    time.sleep(2)

content = aws_json("iam get-credential-report")["Content"]
report = pd.read_csv(io.StringIO(base64.b64decode(content).decode()))
report[["user", "password_enabled", "mfa_active", "access_key_1_active", "access_key_1_last_used_date"]]

💡 The `<root_account>` row is the account owner. If `mfa_active` is `False` on that row, turn MFA on for root **today**.

---
## Part 6 — Policies (UD 03)

### 6.1 What a policy is
A policy is a **JSON** document with rules. Each rule says:

| Field | Question it answers | Example |
|---|---|---|
| `Effect` | Allow or deny? | `"Allow"` or `"Deny"` |
| `Action` | Do what? | `"iam:ListUsers"`, `"s3:GetObject"` |
| `Resource` | On what? | `"*"` (everything) or the ARN of something specific |
| `Condition` *(optional)* | In which situation? | only with MFA, only from one IP… |

**The 3 golden rules of evaluation:**
1. By default, **everything is denied**.
2. An `Allow` opens the door.
3. An explicit `Deny` **always wins**, even if there is an `Allow`.

### 6.2 Policy types

| Type | Who creates it | What it is like |
|---|---|---|
| **AWS managed** | AWS | Ready to use, e.g. `ReadOnlyAccess`. Cannot be edited. |
| **Customer managed** | You | Reusable: can be attached to many users, groups and roles. |
| **Inline** | You | "Glued" to a single user/group/role. If you delete the owner, it goes with it. |

There is also the difference between **identity-based** policies (attached to a user/group/role — everything we see here) and **resource-based** policies (attached to the resource itself, like an S3 bucket policy). The trust policy from Part 3 is a kind of resource-based policy.

Let's look inside a policy managed by AWS:

In [ ]:
arn = "arn:aws:iam::aws:policy/IAMReadOnlyAccess"
version = aws_json(f"iam get-policy --policy-arn {arn}")["Policy"]["DefaultVersionId"]
document = aws_json(f"iam get-policy-version --policy-arn {arn} --version-id {version}")["PolicyVersion"]["Document"]
print(json.dumps(document, indent=2))

### 6.3 Watch golden rule nº 1 in action: everything denied by default
Our test user **has no policy yet**. Can it list the users?

In [ ]:
as_test_user("aws iam list-users --query Users[].UserName --output text")

Denied. It logged in (it has a key), but it has no permission.

### Create our own policy
We want the `readers` group to be able to **see** users, but not **create** them.

In [ ]:
%%writefile read-policy.json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "ReadUsers",
      "Effect": "Allow",
      "Action": ["iam:ListUsers", "iam:GetUser"],
      "Resource": "*"
    }
  ]
}

**Test before applying.** The AWS policy simulator tells you what a policy allows, without creating anything:

In [ ]:
!aws iam simulate-custom-policy --policy-input-list file://read-policy.json --action-names iam:ListUsers iam:CreateUser --query "EvaluationResults[].[EvalActionName,EvalDecision]" --output table

`allowed` = allowed. `implicitDeny` = denied because nothing allowed it (rule nº 1).

Now we create the policy in the account and attach it to the **group** (not to the user — best practice!):

In [ ]:
POLICY_ARN = aws_json(f"iam create-policy --policy-name {POLICY} --policy-document file://read-policy.json")["Policy"]["Arn"]
print(POLICY_ARN)

In [ ]:
!aws iam attach-group-policy --group-name {GROUP} --policy-arn {POLICY_ARN}
!aws iam list-attached-group-policies --group-name {GROUP} --output table

Testing again as `test-student` (we wait a few seconds for AWS to spread the change; if it is still denied, run the cell again):

In [ ]:
time.sleep(15)
print("List users:")
as_test_user("aws iam list-users --query Users[].UserName --output text")
print("Create a user:")
as_test_user("aws iam create-user --user-name intruder")

List: ✅. Create: ⛔. Exactly what the policy says.

### Watch golden rule nº 3: `Deny` always wins
Let's glue an **inline** policy directly onto the user, **denying** the listing:

In [ ]:
%%writefile deny-listing.json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "BlockListing",
      "Effect": "Deny",
      "Action": "iam:ListUsers",
      "Resource": "*"
    }
  ]
}

In [ ]:
!aws iam put-user-policy --user-name {USER} --policy-name test-block --policy-document file://deny-listing.json
time.sleep(15)
as_test_user("aws iam list-users --query Users[].UserName --output text")

Denied — even with the `Allow` coming from the group. **Explicit Deny wins.**

Now we remove the block:

In [ ]:
!aws iam delete-user-policy --user-name {USER} --policy-name test-block

---
## Part 7 — Management and best practices (UD 03 – 3.4)

1. **Do not use root day to day.** Create an administrator user and put root away.
2. **MFA for everyone**, especially root.
3. **Least privilege:** give only what the person needs. It is easy to open more later; it is hard to find out what was done with too much permission.
4. **Permissions per group**, not person by person.
5. **Roles instead of fixed keys** whenever possible (like the EC2 role in Part 3).
6. **Rotate keys** from time to time and disable the ones that are not used.
7. **Validate policies** before applying them and run audits (credential report).

### Practice: validate a policy with a typo
This policy has a hidden mistake. *IAM Access Analyzer* finds it for us:

In [ ]:
%%writefile policy-with-error.json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "iam:ListUser",
      "Resource": "*"
    }
  ]
}

In [ ]:
!aws accessanalyzer validate-policy --policy-type IDENTITY_POLICY --policy-document file://policy-with-error.json --query "findings[].[findingType,issueCode]" --output table

(The mistake: `iam:ListUser` does not exist — the right one is `iam:ListUsers`, with an **s**.)

### Practice: disable a key
Imagine the `test-student` key was leaked. The first step is to **disable** it (no need to delete it straight away):

In [ ]:
!aws iam update-access-key --user-name {USER} --access-key-id {KEY_ID} --status Inactive
time.sleep(10)
as_test_user("aws sts get-caller-identity")

The key stopped working. 🔒

---
## Part 8 — Clean-up

Delete everything we created. **Order matters**: first we remove the links (policies, groups, keys) and only then delete the user, the group, the policy and the role. AWS does not let you delete a user that still has things attached to it.

In [ ]:
cleanup = [
    f"aws iam delete-user-policy --user-name {USER} --policy-name test-block",
    f"aws iam detach-group-policy --group-name {GROUP} --policy-arn {POLICY_ARN}",
    f"aws iam remove-user-from-group --group-name {GROUP} --user-name {USER}",
    f"aws iam delete-access-key --user-name {USER} --access-key-id {KEY_ID}",
    f"aws iam delete-user --user-name {USER}",
    f"aws iam delete-group --group-name {GROUP}",
    f"aws iam delete-policy --policy-arn {POLICY_ARN}",
    f"aws iam detach-role-policy --role-name {ROLE} --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
    f"aws iam delete-role --role-name {ROLE}",
]
for command in cleanup:
    r = subprocess.run(command, shell=True, capture_output=True, text=True)
    status = "✓" if r.returncode == 0 else "– (already gone)"
    print(status, command.split(" --")[0])

---
## Part 9 — Exercises

1. **Using the CLI**, create a group `devs-<your name>` and attach the AWS managed policy `AmazonS3ReadOnlyAccess` to it.
2. Write a policy that allows **only** listing the contents of **one** bucket called `my-class-bucket` (action `s3:ListBucket`). Test it with `simulate-custom-policy` using `--resource-arns arn:aws:s3:::my-class-bucket` and then with another bucket.
3. Using only the CLI's `help`, find out which command lists the policies attached to a **user**.
4. In the console, turn on MFA for your own user. Then confirm it with the CLI using `list-mfa-devices`.
5. At the end, delete everything you created (remember the order!).

<details>
<summary><b>Show answers</b></summary>

**1.**
```
aws iam create-group --group-name devs-joao
aws iam attach-group-policy --group-name devs-joao --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess
```

**2.** Policy:
```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": "s3:ListBucket",
    "Resource": "arn:aws:s3:::my-class-bucket"
  }]
}
```
Test:
```
aws iam simulate-custom-policy --policy-input-list file://bucket-policy.json --action-names s3:ListBucket --resource-arns arn:aws:s3:::my-class-bucket
aws iam simulate-custom-policy --policy-input-list file://bucket-policy.json --action-names s3:ListBucket --resource-arns arn:aws:s3:::other-bucket
```
The first gives `allowed`, the second `implicitDeny`.

**3.** `aws iam list-attached-user-policies --user-name <name>` (and, for inline policies, `aws iam list-user-policies`).

**4.** `aws iam list-mfa-devices --user-name <your user>`

**5.** Detach the policy from the group (`detach-group-policy`) and then `delete-group`.
</details>

---
## CLI cheat sheet (IAM)

| I want to… | Command |
|---|---|
| Know who I am | `aws sts get-caller-identity` |
| Set up my keys | `aws configure` |
| List users | `aws iam list-users` |
| Create a user / group | `aws iam create-user --user-name X` / `aws iam create-group --group-name X` |
| Put a user in a group | `aws iam add-user-to-group --group-name G --user-name U` |
| Create a role | `aws iam create-role --role-name R --assume-role-policy-document file://trust.json` |
| Create a policy | `aws iam create-policy --policy-name P --policy-document file://policy.json` |
| Attach a policy to a group / user / role | `attach-group-policy` / `attach-user-policy` / `attach-role-policy` |
| Test a policy without applying it | `aws iam simulate-custom-policy --policy-input-list file://p.json --action-names ...` |
| Create / list / disable a key | `create-access-key` / `list-access-keys` / `update-access-key --status Inactive` |
| Check MFA | `aws iam list-mfa-devices --user-name U` |
| Get help | `aws iam help` · `aws iam <action> help` |